# TuringSQL 3/3: merge the trained E3 adapter and run all evaluations (T4)

Training (notebook 2) finished but its merge and evaluation steps crashed (a `torchao` version clash in
`peft`, and one malformed model-written SQL string that crashed the scorer). Both are fixed in the code
package, so this notebook only **merges, verifies and evaluates**; no training.

**Settings:** Accelerator **GPU T4** (x1 or x2), Internet **On**.
**Inputs:** the **new version** of the `turingsql-kaggle` dataset (with the fixes) **and** the output of
notebook 2 (*Add Input -> Your Work -> turingsql-2-train-E3*). Everything is saved in this notebook's **Output**.

A failed evaluation no longer stops the notebook: it is listed at the end, and re-running skips finished ones.

In [ ]:
# Code + data are uploaded as a private Kaggle dataset built by scripts/package_for_kaggle.py
import glob
import json
import os
import subprocess
import sys
import zipfile
import shutil

WORK = "/kaggle/working/TuringSQL"
if not os.path.isdir(WORK):
    # Kaggle usually unzips uploaded archives; handle both the zip and the unpacked folder.
    zips = glob.glob("/kaggle/input/**/turingsql_kaggle.zip", recursive=True)
    trees = [os.path.dirname(os.path.dirname(p))
             for p in glob.glob("/kaggle/input/**/turingsql/__init__.py", recursive=True)]

    def built(tree):  # newest package wins; copies without a stamp (older code) lose
        f = os.path.join(tree, "PACKAGE_INFO.json")
        return json.load(open(f))["built_utc"] if os.path.isfile(f) else ""

    trees.sort(key=built, reverse=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif trees:
        print("code from", trees[0], built(trees[0]) or "(no build stamp)")
        shutil.copytree(trees[0], WORK)
    else:
        raise FileNotFoundError("attach the turingsql-kaggle dataset (dist/turingsql_kaggle.zip)")
os.chdir(WORK)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi
!pip -q install "transformers>=4.45,<5" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.0" "sqlglot>=25" rank_bm25
!pip -q uninstall -y torchao
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "no GPU: set the Accelerator to GPU T4"

In [ ]:
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
FT = "/kaggle/working/turingsql-1.5b"
R = "/kaggle/working/results"
os.makedirs(R, exist_ok=True)
adapters = glob.glob("/kaggle/input/**/ckpt/E3_seed0/final_adapter/adapter_model.safetensors", recursive=True)
assert adapters, "attach the output of notebook 2 (turingsql-2-train-E3) as an input"
ADAPTER = os.path.dirname(adapters[0])
print("adapter:", ADAPTER)
# keep the E0 Spider result already produced by notebook 2 (do not redo 12 minutes of work)
for d in glob.glob("/kaggle/input/**/results/E0_qwen15b_hf_spider-dev", recursive=True):
    if os.path.isfile(os.path.join(d, "summary.json")) and not os.path.isdir(f"{R}/{os.path.basename(d)}"):
        shutil.copytree(d, f"{R}/{os.path.basename(d)}"); print("reused", d)
# the BIRD/Spider SFT files are only needed for the 50-prompt merge check
reports = glob.glob("/kaggle/input/**/data/processed/sft_val.jsonl", recursive=True)
VAL = reports[0] if reports else "data/processed/sft_val.jsonl"
print("merge-check prompts:", VAL)

In [ ]:
# Merge the adapter into fp16 weights and check greedy outputs are unchanged. Stops the notebook on failure.
r = subprocess.run(f"python scripts/merge.py --adapter {ADAPTER} --out {FT} --check-prompts {VAL} --n-check 50 "
                   "--dtype float16 --device cuda", shell=True, capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-3000:])
assert os.path.isfile(f"{FT}/model.safetensors") or glob.glob(f"{FT}/model*.safetensors"), "merge failed, see above"
print(open(f"{FT}/merge_report.json").read())

In [ ]:
RUNS = [
    # (run tag, model, extra evaluate.py flags)
    ("E3_seed0", FT, ""),                     # the main result first, so it exists even if time runs out
    ("E0_qwen15b", BASE, ""),                 # zero-shot base
    ("E1_qwen15b_3shot", BASE, "--shots 3"),  # 3 retrieved Spider-train examples
]
FAILED = []
def evaluate(tag, model, flags, ds):
    run_id = f"{tag}_hf_{ds}"
    if os.path.isfile(f"{R}/{run_id}/summary.json"):
        print("skip (done):", run_id); return
    cmd = (f"python scripts/evaluate.py --engine hf --model {model} --device cuda --dtype float16 "
           f"--dataset {ds} {'--limit 500' if ds != 'olist-dev' else ''} {flags} --run-id {run_id} --out {R}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        FAILED.append(run_id); print("FAILED:", run_id); print(r.stderr[-2500:])
    else:
        print(open(f"{R}/{run_id}/summary.md").read())

for tag, model, flags in RUNS:
    for ds in ["spider-dev", "bird-dev", "olist-dev"]:
        evaluate(tag, model, flags, ds)
evaluate("E5_seed0_glossary", FT, "--definitions", "olist-dev")
evaluate("E5_seed0_evidence", FT, "--definitions", "bird-dev")
print("FAILED runs:", FAILED or "none")

In [ ]:
pairs = [(f"{a}_hf_{ds}", f"E3_seed0_hf_{ds}") for ds in ["spider-dev", "bird-dev", "olist-dev"]
         for a in ["E0_qwen15b", "E1_qwen15b_3shot"]]
pairs += [("E3_seed0_hf_olist-dev", "E5_seed0_glossary_hf_olist-dev"),
          ("E3_seed0_hf_bird-dev", "E5_seed0_evidence_hf_bird-dev")]
for a, b in pairs:
    if os.path.isdir(f"{R}/{a}") and os.path.isdir(f"{R}/{b}"):
        subprocess.run(f"python scripts/compare.py {R}/{a} {R}/{b}", shell=True)
        print()

In [ ]:
os.chdir("/kaggle/working")
!zip -qr results.zip results && ls -la results.zip
!rm -rf TuringSQL
!du -sh /kaggle/working/* | cat